# 07 — Detecção de anomalias

**Objetivo:** marcar dias cujo comportamento operacional foge do padrão da própria entidade,
olhando várias dimensões ao mesmo tempo (volume, severidade, OLA, MTTR e alcance).

**Algoritmo:** Isolation Forest (não supervisionado).

**Por que não há precisão/recall aqui:** o dataset não tem rótulo de "dia anômalo". Inventar uma AUC
seria pior que não ter. O que se mede é a **concordância com o z-score móvel de 21 dias** (régua
univariada): onde os dois apontam juntos, a evidência é forte; onde divergem, o Isolation Forest
costuma estar vendo uma combinação de fatores que uma variável sozinha não mostra.

> Pré-requisito: executar antes o `02_engenharia_de_features.ipynb`.

In [ ]:
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

PASTA = Path.cwd()
DADOS = PASTA / 'dados'            # saída do notebook 02, lida pelos notebooks de modelo
ARTEFATOS = PASTA / 'artefatos'    # modelos treinados por estes notebooks
DADOS.mkdir(exist_ok=True)
ARTEFATOS.mkdir(exist_ok=True)

SEMENTE = 42

base = pd.read_pickle(DADOS / 'base.pkl')
print('Base carregada:', f"{len(base['incidentes']):,} incidentes |",
      f"regime a partir de {base['inicio_regime']:%d/%m/%Y}")

## 1. Features de comportamento diário

In [ ]:
COLUNAS = [
    'lag_0', 'media_7', 'razao_7_28', 'var_7_vs_7', 'desvio_7', 'cv_28',
    'criticos_media_7', 'share_criticos_7', 'p2_media_7', 'violacoes_media_7',
    'taxa_violacao_media_7', 'mttr_medio_media_7', 'delta_mttr',
    'itens_distintos_media_7', 'familias_distintas_media_7',
]
quadro = base['contexto'].dropna(subset=COLUNAS).reset_index(drop=True)
X = quadro[COLUNAS].replace([np.inf, -np.inf], np.nan).fillna(0.0)
print(f'Dias × entidades avaliados: {len(X):,} | features: {len(COLUNAS)}')
X.describe().T.round(2)

## 2. Treino

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

CONTAMINACAO = 0.05

escala = StandardScaler().fit(X)
modelo = IsolationForest(n_estimators=300, contamination=CONTAMINACAO, random_state=SEMENTE, n_jobs=-1)
modelo.fit(escala.transform(X))

pontuacao = -modelo.score_samples(escala.transform(X))     # maior = mais anômalo
limiar = float(np.quantile(pontuacao, 1 - CONTAMINACAO))
print(f'Limiar de pontuação (percentil {1 - CONTAMINACAO:.0%}): {limiar:.4f}')

fig, eixo = plt.subplots()
eixo.hist(pontuacao, bins=80, color='#2563eb')
eixo.axvline(limiar, color='#dc2626', ls='--', label='limiar')
eixo.set_title('Distribuição da pontuação de anomalia'); eixo.legend(); plt.show()

## 3. Avaliação: concordância com o z-score móvel de 21 dias

In [ ]:
z = (quadro.groupby(['dimensao', 'chave'])['lag_0']
     .transform(lambda s: (s - s.rolling(21, min_periods=10).mean())
                / s.rolling(21, min_periods=10).std().replace(0, np.nan)))
marcado_if = pontuacao >= limiar
marcado_z = (z.abs() >= 2.5).fillna(False).to_numpy()
ambos = int((marcado_if & marcado_z).sum())

pd.Series({
    'dias avaliados': len(quadro),
    'marcados pelo Isolation Forest': int(marcado_if.sum()),
    'marcados pelo z-score (|z| ≥ 2,5)': int(marcado_z.sum()),
    'marcados pelos dois': ambos,
    'concordância (dos marcados pelo IF)': round(ambos / max(int(marcado_if.sum()), 1), 3),
})

In [ ]:
# Onde o Isolation Forest vê anomalia que o z-score não vê: perfil médio desses dias vs. o normal.
so_if = marcado_if & ~marcado_z
comparativo = pd.DataFrame({
    'normal': X[~marcado_if].mean(),
    'só Isolation Forest': X[so_if].mean(),
    'ambos': X[marcado_if & marcado_z].mean(),
}).round(3)
comparativo

In [ ]:
# Série global com os dias marcados.
global_ = quadro['dimensao'] == 'global'
fig, eixo = plt.subplots()
eixo.plot(quadro.loc[global_, 'data'], quadro.loc[global_, 'lag_0'], color='#94a3b8', lw=0.9, label='volume elegível')
eixo.scatter(quadro.loc[global_ & marcado_if, 'data'], quadro.loc[global_ & marcado_if, 'lag_0'],
             color='#dc2626', s=18, label='anomalia (Isolation Forest)', zorder=3)
eixo.set_title('Série global — dias marcados como anômalos'); eixo.legend(); plt.show()

## 4. Artefato

In [ ]:
import joblib

joblib.dump({'modelo': modelo, 'escala': escala, 'colunas': COLUNAS, 'limiar': limiar},
            ARTEFATOS / 'model_anomalia.joblib', compress=3)
print('Artefato gravado em', ARTEFATOS / 'model_anomalia.joblib')